In [2]:
# --- Core Python ---
import os

# --- Numerical & Data Analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns
import re

# --- Signal Processing ---
from scipy.signal import butter, filtfilt, resample, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Specialized Neuro Tools ---
import neurokit2 as nk

print("✅ Imports loaded")

# --- Pandas display settings (optional) ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)


✅ Imports loaded


In [3]:
# --- Respiration file paths (.h5) --- # Baseline recordings
resp_paths_bl = {
    "BL_1_1_d1_2": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_1_d1_2_20250623_103713_merged.h5",
    "BL_1_2_sub1_1": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_2_sub_1_1_20250623_120135_merged.h5",
    "BL_2_3_d2_4": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_3_d2_4_20250623_145448_merged.h5",
    "BL_2_4_sub2_3": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_4_sub2_3_20250623_141419_merged.h5",
    "BL_3_5_d3_6": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_5_d3_6_20250623_154154_merged.h5",
    "BL_3_6_sub3_5": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_6_sub3_5_20250623_172635_merged.h5",
    "BL_4_7_d4_8": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_7_d4_8_20250623_185042_merged.h5",
    "BL_4_8_sub4_7": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_8_sub4_7_20250623_180810_merged.h5"
}
resp_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5",
    "CM_2_4_sub2_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5",
}
# --- BORIS annotation file paths (.csv) --- # Baseline recordings
boris_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_1_d1_2_20250623_111352.1.csv",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_2_sub1_1_20250623_133932.1_VT.csv",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_3_d2_4_20250623_151153.1.csv",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_4_sub2_3_20250623_143348.1_VT.csv",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_5_d3_6_20250623_160001.csv",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_6_sub3_5_20250623_174348.csv",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_7_d4_8_20250623_193718.csv",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_8_sub4_7_20250623_182649.1_VT.csv",
}

print(f"Resp files: {len(resp_paths_cm)} trials")
print(f"Resp files: {len(resp_paths_bl)} trials")
print(f"BORIS files: {len(boris_paths_cm)} trials")

Resp files: 8 trials
Resp files: 8 trials
BORIS files: 8 trials


In [4]:
# --- Respiration file paths (.h5) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

# --- BORIS annotation file paths (.csv) ---
boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

print(f"Resp files: {len(resp_paths)} trials")
print(f"BORIS files: {len(boris_paths)} trials")

Resp files: 23 trials
BORIS files: 15 trials


In [5]:
import h5py

# Pick any one of your respiration .h5 files
example_file = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5"

with h5py.File(example_file, "r") as f:
    print("Top-level keys in this file:")
    print(list(f.keys()))


Top-level keys in this file:
['ekg', 'ekg_metadata', 'metadata', 'resp', 'resp_metadata']


In [58]:
import h5py, os

def inspect_h5_structure(file_path):
    """Print top-level keys and their attributes (if any) for a given .h5 file."""
    print(f"\n📂 {os.path.basename(file_path)}")
    try:
        with h5py.File(file_path, "r") as f:
            keys = list(f.keys())
            print("  ├─ Top-level keys:", keys)

            # Print attribute info for each key
            for key in keys:
                if isinstance(f[key], h5py.Group) and f[key].attrs:
                    print(f"  │  ├─ Attributes in {key}:")
                    for k, v in f[key].attrs.items():
                        print(f"  │  │    {k}: {v}")
    except Exception as e:
        print(f"  ❌ Error: {e}")


In [59]:
# Try one RI and one CM file
inspect_h5_structure(resp_paths["RI1_3_6"])
inspect_h5_structure(resp_paths_cm["CM_3_5_d3_6"])



📂 RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5
  ├─ Top-level keys: ['ekg', 'ekg_metadata', 'metadata', 'resp', 'resp_metadata']
  │  ├─ Attributes in ekg_metadata:
  │  │    channel_id: 21
  │  │    duration_sec: 603.2723
  │  │    num_samples: 12065446
  │  │    sampling_frequency: 20000.0
  │  │    stream_id: trodes
  │  ├─ Attributes in metadata:
  │  │    date: 2025-06-21
  │  │    negative_agent_id: RB3
  │  │    positive_agent_id: 5.3
  │  │    subject_id: 3.6
  │  │    time: 12:53:12
  │  │    trial_type: RI1
  │  ├─ Attributes in resp_metadata:
  │  │    duration_sec: 603.2723
  │  │    sampling_frequency: 20000.0

📂 CM_s3_5_d3_6_20250623_170708_merged.h5
  ├─ Top-level keys: ['ekg_metadata', 'metadata', 'resp', 'resp_metadata']
  │  ├─ Attributes in resp_metadata:
  │  │    byte_order: little endian
  │  │    channel: analog_ECU_Ain1
  │  │    duration_sec: 605.9417
  │  │    duration_sec_from_ekg_sr: 605.9417
  │  │    duration_sec_hardcoded_20k: 605.9417
  │  │    num_samp

In [60]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Load respiration signal from .h5 file, clean, and resample to a uniform rate.
    Uses 'resp' dataset and 'resp_metadata' for sampling information.
    Returns (resp_cleaned, time_vector, fs_original)
    """
    try:
        # --- Open the .h5 file ---
        with h5py.File(h5_file, "r") as f:
            if "resp" not in f:
                print(f"⚠️ No 'resp' dataset found in {os.path.basename(h5_file)}")
                return None, None, None

            # --- Load respiration data ---
            resp = np.array(f["resp"]).flatten()

            # --- Load metadata ---
            fs = None
            if "resp_metadata" in f:
                meta = dict(f["resp_metadata"].attrs)
                fs = float(meta.get("sampling_frequency", 0))
                duration_sec = float(meta.get("duration_sec", len(resp)/fs if fs else 0))
            else:
                fs = 20000.0  # fallback if metadata missing
                duration_sec = len(resp) / fs

        # --- Filter (0.1–15 Hz band) and resample to target_rate ---
        b, a = butter(4, [0.1, 15.0], btype="bandpass", fs=fs)
        filtered = filtfilt(b, a, resp)
        N_out = int(round(duration_sec * target_rate))
        resampled = resample(filtered, N_out)

        # --- Build time vector ---
        t = np.arange(len(resampled)) / target_rate

        return resampled, t, fs  # return cleaned signal, time vector, and original fs

    except Exception as e:
        print(f"❌ Error loading {os.path.basename(h5_file)}: {e}")
        return None, None, None


In [71]:
# --- Define all respiration sets ---
resp_sets = {
    "CM": resp_paths_cm,
    "BL": resp_paths_bl,
    "RI": resp_paths
}

# --- Dictionary to hold all results ---
resp_data_all = {}

# --- Loop through each condition set ---
for set_name, paths in resp_sets.items():
    print(f"\n==============================")
    print(f"🔹 Processing {set_name} respiration files...")
    print(f"==============================")

    for label, path in paths.items():
        try:
            resp_clean, t, fs_original = load_clean_resp_signal(path, target_rate=100)

            if resp_clean is not None:
                effective_fs = len(resp_clean) / t[-1] if len(t) > 1 else np.nan
                duration = t[-1]

                resp_data_all[label] = {
                    "signal": resp_clean,
                    "time": t,
                    "fs_original": fs_original,
                    "fs_resampled": effective_fs,
                    "duration_sec": duration,
                    "condition": set_name
                }

                print(
                    f"✅ Loaded {label}: {len(resp_clean)} samples "
                    f"(duration={duration:.1f}s, original fs={fs_original:.0f} Hz → resampled to {effective_fs:.0f} Hz)"
                )
            else:
                print(f"⚠️ Skipped {label} (no signal found)")

        except Exception as e:
            print(f"❌ Error processing {label}: {e}")

    print(f"✅ Finished {set_name}: {len(paths)} files\n")

print(f"🎯 Total loaded: {len(resp_data_all)} respiration traces")




🔹 Processing CM respiration files...
✅ Loaded CM_1_1_d1_2: 61262 samples (duration=612.6s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_1_2_sub1_1: 60566 samples (duration=605.6s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_2_3_d2_4: 61143 samples (duration=611.4s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_2_4_sub2_2: 62196 samples (duration=622.0s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_3_5_d3_6: 60594 samples (duration=605.9s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_3_6_sub3_5: 60426 samples (duration=604.2s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_4_7_d4_8: 60319 samples (duration=603.2s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded CM_4_8_sub4_7: 60508 samples (duration=605.1s, original fs=20000 Hz → resampled to 100 Hz)
✅ Finished CM: 8 files


🔹 Processing BL respiration files...
✅ Loaded BL_1_1_d1_2: 62369 samples (duration=623.7s, original fs=20000 Hz → resampled to 100 Hz)
✅ Loaded B

In [62]:
# ======================================================
# 📦 Load Respiration Data (Cagemate, Resident–Intruder, Baseline)
# ======================================================

import os

# --- Helper function: Load all .h5 respiration files ---
def load_all_resp(resp_paths, label_prefix=""):
    """Load all respiration traces from provided path dictionary."""
    resp_data = {}
    for label, path in resp_paths.items():
        print(f"Processing {label_prefix}{label}...")
        signal, time, rate = load_clean_resp_signal(path)

        if signal is not None:
            resp_data[label] = {
                "signal": signal,
                "time": time,
                "sampling_rate": rate
            }
            print(f"✅ Loaded {label}: {len(signal)} samples, duration={time[-1]:.1f}s, fs={rate} Hz")
        else:
            print(f"❌ Skipped {label} (failed to load)")

    print(f"\n✅ Finished: {len(resp_data)} respiration traces loaded\n")
    return resp_data


# ======================================================
# 🧠 Patch RI files (metadata copy from ekg_metadata → resp_metadata)
# ======================================================

import h5py

def copy_ekg_to_resp_metadata(h5_path):
    """Copies sampling info from ekg_metadata → resp_metadata if missing."""
    try:
        with h5py.File(h5_path, "a") as f:
            if "ekg_metadata" not in f:
                return
            ekg_meta = f["ekg_metadata"].attrs
            if "resp_metadata" not in f:
                f.create_group("resp_metadata")
            resp_meta = f["resp_metadata"].attrs

            # Copy attributes only if they don't exist
            for key in ["sampling_frequency", "duration_sec"]:
                if key in ekg_meta and key not in resp_meta:
                    resp_meta[key] = ekg_meta[key]
                    print(f"✅ Copied {key}={ekg_meta[key]} to resp_metadata ({os.path.basename(h5_path)})")
    except Exception as e:
        print(f"⚠️ Could not patch {os.path.basename(h5_path)}: {e}")


In [107]:
# ======================================================
# Apply metadata fix to RI respiration files
# ======================================================
for path in resp_paths.values():
    copy_ekg_to_resp_metadata(path)

# ======================================================
# Load all respiration sets
# ======================================================
resp_data_cm = load_all_resp(resp_paths_cm, "CM_")  # Cagemate
resp_data_ri = load_all_resp(resp_paths, "RI_")     # Resident–Intruder (Juvenile / CD1)
resp_data_bl = load_all_resp(resp_paths_bl, "BL_")  # Baseline

print(f"Respiration loaded: CM={len(resp_data_cm)}, RI={len(resp_data_ri)}, BL={len(resp_data_bl)}")



Processing CM_CM_1_1_d1_2...
✅ Loaded CM_1_1_d1_2: 61262 samples, duration=612.6s, fs=20000.0 Hz
Processing CM_CM_1_2_sub1_1...
✅ Loaded CM_1_2_sub1_1: 60566 samples, duration=605.6s, fs=20000.0 Hz
Processing CM_CM_2_3_d2_4...
✅ Loaded CM_2_3_d2_4: 61143 samples, duration=611.4s, fs=20000.0 Hz
Processing CM_CM_2_4_sub2_2...
✅ Loaded CM_2_4_sub2_2: 62196 samples, duration=622.0s, fs=20000.0 Hz
Processing CM_CM_3_5_d3_6...
✅ Loaded CM_3_5_d3_6: 60594 samples, duration=605.9s, fs=20000.0 Hz
Processing CM_CM_3_6_sub3_5...
✅ Loaded CM_3_6_sub3_5: 60426 samples, duration=604.2s, fs=20000.0 Hz
Processing CM_CM_4_7_d4_8...
✅ Loaded CM_4_7_d4_8: 60319 samples, duration=603.2s, fs=20000.0 Hz
Processing CM_CM_4_8_sub4_7...
✅ Loaded CM_4_8_sub4_7: 60508 samples, duration=605.1s, fs=20000.0 Hz

✅ Finished: 8 respiration traces loaded

Processing RI_RI1_3_6...
✅ Loaded RI1_3_6: 60327 samples, duration=603.3s, fs=20000.0 Hz
Processing RI_RI2_3_6...
✅ Loaded RI2_3_6: 61000 samples, duration=610.0s, fs

In [72]:
signal, t, fs = load_clean_resp_signal(resp_paths["RI1_3_6"])
print(f"Length: {len(signal)}, Duration: {t[-1]:.1f}s, Effective Fs: {len(signal)/t[-1]:.1f} Hz")


Length: 60327, Duration: 603.3s, Effective Fs: 100.0 Hz


In [73]:
# ======================================================
# 📑 Load BORIS behavioral annotations (Cagemate)
# ======================================================

boris_data_cm = {}

for label, path in boris_paths_cm.items():
    try:
        df = pd.read_csv(path)
        boris_data_cm[label] = df
        print(f"✅ Loaded {label}: {df.shape[0]} rows")
    except Exception as e:
        print(f"❌ Failed to load {label}: {e}")

print(f"\n✅ Finished loading {len(boris_data_cm)} BORIS files")


✅ Loaded CM_1_1_d1_2: 129 rows
✅ Loaded CM_1_2_sub1_1: 41 rows
✅ Loaded CM_2_3_d2_4: 97 rows
✅ Loaded CM_2_4_sub2_3: 75 rows
✅ Loaded CM_3_5_d3_6: 90 rows
✅ Loaded CM_3_6_sub3_5: 0 rows
✅ Loaded CM_4_7_d4_8: 49 rows
✅ Loaded CM_4_8_sub4_7: 70 rows

✅ Finished loading 8 BORIS files


In [82]:
# ============================================
# 📂 Load ALL BORIS annotation CSVs (BL, CM, RI, etc.)
# ============================================

# Create a combined dictionary of all BORIS sets that actually exist in memory
boris_sets = {}

# Check each possible dictionary name and add if defined
if 'boris_paths_bl' in globals():
    boris_sets['BL'] = boris_paths_bl
if 'boris_paths_cm' in globals():
    boris_sets['CM'] = boris_paths_cm
if 'boris_paths' in globals():
    boris_sets['RI'] = boris_paths  # includes RI1, RI2, BLRI if present

print(f"📁 Found BORIS sets: {list(boris_sets.keys())}")

# Main storage dict
boris_data_all = {}

# --- Loop through and load each set ---
for condition_name, paths_dict in boris_sets.items():
    print(f"\n==============================")
    print(f"🔹 Loading BORIS {condition_name} files...")
    print("==============================")

    for label, csv_path in paths_dict.items():
        try:
            df = pd.read_csv(csv_path)
            boris_data_all[label] = {
                "data": df.copy(),
                "condition": condition_name
            }
            print(f"✅ Loaded {label}: {df.shape[0]} rows")

        except Exception as e:
            print(f"❌ Error loading {label}: {e}")

print(f"\n🎯 Total BORIS trials loaded: {len(boris_data_all)}")


📁 Found BORIS sets: ['CM', 'RI']

🔹 Loading BORIS CM files...
✅ Loaded CM_1_1_d1_2: 129 rows
✅ Loaded CM_1_2_sub1_1: 41 rows
✅ Loaded CM_2_3_d2_4: 97 rows
✅ Loaded CM_2_4_sub2_3: 75 rows
✅ Loaded CM_3_5_d3_6: 90 rows
✅ Loaded CM_3_6_sub3_5: 0 rows
✅ Loaded CM_4_7_d4_8: 49 rows
✅ Loaded CM_4_8_sub4_7: 70 rows

🔹 Loading BORIS RI files...
✅ Loaded RI1_3_6: 37 rows
✅ Loaded RI2_3_6: 109 rows
✅ Loaded RI1_4_7: 37 rows
✅ Loaded RI2_4_7: 108 rows
✅ Loaded RI1_2_3: 29 rows
✅ Loaded RI2_2_3: 82 rows
✅ Loaded RI1_4_8: 55 rows
✅ Loaded RI2_4_8: 113 rows
✅ Loaded RI1_1_1: 92 rows
✅ Loaded RI2_1_1: 37 rows
✅ Loaded RI1_1_2: 89 rows
✅ Loaded RI2_1_2: 58 rows
✅ Loaded RI1_2_4: 69 rows
✅ Loaded RI2_2_4: 77 rows
✅ Loaded RI2_3_5: 27 rows

🎯 Total BORIS trials loaded: 23


In [85]:
resp_paths_cm_fixed = resp_paths_cm.copy()
resp_paths_cm_fixed["CM_2_4_sub2_3"] = resp_paths_cm_fixed.pop("CM_2_4_sub2_2")


In [87]:
print("🔹 Respiration CM keys:")
for k in sorted(resp_paths_cm_fixed.keys()):
    print(" ", repr(k))

print("\n🔹 BORIS CM keys:")
for k in sorted(boris_paths_cm.keys()):
    print(" ", repr(k))

print("\n🔍 Keys in BORIS not in Resp:")
print(set(boris_paths_cm.keys()) - set(resp_paths_cm_fixed.keys()))

print("\n🔍 Keys in Resp not in BORIS:")
print(set(resp_paths_cm_fixed.keys()) - set(boris_paths_cm.keys()))


🔹 Respiration CM keys:
  'CM_1_1_d1_2'
  'CM_1_2_sub1_1'
  'CM_2_3_d2_4'
  'CM_2_4_sub2_3'
  'CM_3_5_d3_6'
  'CM_3_6_sub3_5'
  'CM_4_7_d4_8'
  'CM_4_8_sub4_7'

🔹 BORIS CM keys:
  'CM_1_1_d1_2'
  'CM_1_2_sub1_1'
  'CM_2_3_d2_4'
  'CM_2_4_sub2_3'
  'CM_3_5_d3_6'
  'CM_3_6_sub3_5'
  'CM_4_7_d4_8'
  'CM_4_8_sub4_7'

🔍 Keys in BORIS not in Resp:
set()

🔍 Keys in Resp not in BORIS:
set()


In [86]:
# Combine all respiration keys across BL, CM, and RI
resp_keys_all = set()
for dname in ['resp_paths_bl', 'resp_paths_cm', 'resp_paths']:
    if dname in globals():
        resp_keys_all |= set(globals()[dname].keys())

boris_keys_all = set(boris_data_all.keys())

print("\nBORIS-only (no matching respiration):", boris_keys_all - resp_keys_all)
print("Resp-only (no matching BORIS):", resp_keys_all - boris_keys_all)
print(f"\n✅ Matched trials: {len(boris_keys_all & resp_keys_all)}")



BORIS-only (no matching respiration): {'CM_2_4_sub2_3'}
Resp-only (no matching BORIS): {'BLRI_2_4', 'BLRI_4_7', 'BLRI_1_1', 'CM_2_4_sub2_2', 'BL_3_6_sub3_5', 'BL_1_2_sub1_1', 'BLRI_3_6', 'BLRI_2_3', 'BL_2_3_d2_4', 'BL_4_8_sub4_7', 'RI1_3_5', 'BLRI_1_2', 'BL_4_7_d4_8', 'BL_1_1_d1_2', 'BL_3_5_d3_6', 'BLRI_4_8', 'BL_2_4_sub2_3'}

✅ Matched trials: 22


In [74]:
print("BORIS CM keys:")
print(sorted(list(boris_data_cm.keys())))

print("\nRespiration CM keys:")
print(sorted(list(resp_data_cm.keys())))


BORIS CM keys:
['CM_1_1_d1_2', 'CM_1_2_sub1_1', 'CM_2_3_d2_4', 'CM_2_4_sub2_3', 'CM_3_5_d3_6', 'CM_3_6_sub3_5', 'CM_4_7_d4_8', 'CM_4_8_sub4_7']

Respiration CM keys:
['CM_1_1_d1_2', 'CM_1_2_sub1_1', 'CM_2_3_d2_4', 'CM_2_4_sub2_2', 'CM_3_5_d3_6', 'CM_3_6_sub3_5', 'CM_4_7_d4_8', 'CM_4_8_sub4_7']


In [88]:
# ======================================================
# ✅ Final key normalization for Cagemate trials
# ======================================================

def fix_cm_key(k):
    """Standardize CM trial names so BORIS & Resp match."""
    # Unify prefix (some Resp files have 'CM_CM_')
    k = k.replace("CM_CM_", "CM_")
    # Fix typo: sub2_2 -> sub2_3
    k = k.replace("sub2_2", "sub2_3")
    return k

# Apply to both datasets
resp_data_cm_fixed  = {fix_cm_key(k): v for k, v in resp_data_cm.items()}
boris_data_cm_fixed = {fix_cm_key(k): v for k, v in boris_data_cm.items()}



In [101]:
def summarize_resp(resp_dict):
    """
    Summarize respiration data for each trial in resp_dict using the same
    peak detection parameters as sniff-window analysis.
    
    Uses:
      - min distance = 0.083 s (≈12 Hz max)
      - fs ≈ 100 Hz resampled rate
    Computes:
      - Resp_MeanRate_Hz
      - Resp_IBI_Mean_s
      - Resp_IBI_CV
    """
    summary = []

    for trial, d in resp_dict.items():
        sig = d["signal"]
        fs = d.get("fs_resampled", d.get("fs_original", None))
        if fs is None:
            print(f"⚠️ Missing sampling rate for {trial}")
            continue

        # --- Peak detection (≥12 Hz refractory → 0.083 s) ---
        peaks, _ = find_peaks(sig, distance=fs * 0.0833)
        if len(peaks) < 2:
            print(f"⚠️ Too few peaks detected in {trial}")
            continue

        # --- Compute IBI and summary metrics ---
        time = np.arange(len(sig)) / fs
        ibi_values = np.diff(time[peaks])
        rate_hz = len(peaks) / (len(sig) / fs)
        ibi_mean = np.mean(ibi_values)
        ibi_cv = np.std(ibi_values) / np.mean(ibi_values)

        summary.append({
            "Trial": trial,
            "Resp_MeanRate_Hz": rate_hz,
            "Resp_IBI_Mean_s": ibi_mean,
            "Resp_IBI_CV": ibi_cv,
            "Resp_nPeaks": len(peaks),
            "Resp_fs": fs
        })

    return pd.DataFrame(summary)



In [102]:
def summarize_boris(boris_dict, condition_label="Cagemate", rank_map=None):
    """
    Summarize BORIS behavioral annotations for each trial.
    Returns a DataFrame with total time per behavior and metadata.
    """
    summary = []
    for trial, df in boris_dict.items():
        if "Behavior" not in df.columns or "Duration" not in df.columns:
            continue

        behavior_durations = (
            df.groupby("Behavior")["Duration"].sum()
            .reset_index()
            .rename(columns={"Duration": "Total_Duration_s"})
        )
        total_time = behavior_durations["Total_Duration_s"].sum()

        summary.append({
            "Trial": trial,
            "Condition": condition_label,
            "Total_BehaviorTime_s": total_time,
            "n_Behaviors": len(behavior_durations)
        })

    return pd.DataFrame(summary)


In [96]:
first_key = list(resp_data_all.keys())[0]
print("Example trial key:", first_key)
print("Available keys in resp_data_all entry:")
print(resp_data_all[first_key].keys())


Example trial key: CM_1_1_d1_2
Available keys in resp_data_all entry:
dict_keys(['signal', 'time', 'fs_original', 'fs_resampled', 'duration_sec', 'condition'])


In [91]:
# --- Re-run summaries ---
resp_summary_cm  = summarize_resp(resp_data_cm_fixed)
boris_summary_cm = summarize_boris(boris_data_cm_fixed, "Cagemate", rank_fix)

# --- Merge (outer keeps all valid trials) ---
merged_cm = pd.merge(boris_summary_cm, resp_summary_cm, on="Trial", how="outer")
print("✅ CM Trials now in merged_cm:")
print(sorted(merged_cm["Trial"].tolist()))


KeyError: 'Trial'

In [ ]:
# 🧹 Drop duplicate trial manually
merged_cm = merged_cm.drop_duplicates(subset="Trial", keep="first").reset_index(drop=True)

print(f"✅ Cleaned CM trials: {len(merged_cm)} total")
print(sorted(merged_cm['Trial'].tolist()))



In [ ]:
merged_all = pd.concat([merged_cm, merged_ri], ignore_index=True)
print(f"✅ merged_all shape: {merged_all.shape}")


In [ ]:
print(merged_all)


In [ ]:
# --- Keep trials with known social context ---
valid_conditions = ["Dominant", "Subordinate", "Juvenile", "CD1"]
df_multi = merged_all[merged_all["Condition"].isin(valid_conditions)].copy()

print(f"✅ Total labeled trials: {len(df_multi)}")
print(df_multi["Condition"].value_counts())


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
import pandas as pd

# Encode target
le = LabelEncoder()
y_enc = le.fit_transform(df_multi["Condition"])

# Features
feature_cols = [
    "allogrooming", "anogenital sniffing", "body sniffing",
    "chasing", "facial sniffing", "fighting", "Resp_MeanRate"
]
X = df_multi[feature_cols].fillna(0)

# Model
model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, multi_class="multinomial"))

scores = cross_val_score(model, X, y_enc, cv=5, scoring="accuracy")

print(f"✅ Mean 5-fold CV accuracy: {scores.mean():.2f} ± {scores.std():.2f}")
print("\nLabel mapping:")
for idx, cls in enumerate(le.classes_):
    print(f"  {idx}: {cls}")


In [ ]:
model.fit(X, y_enc)
coefs = model.named_steps["logisticregression"].coef_
coef_df = pd.DataFrame(coefs.T, index=feature_cols, columns=le.classes_)
display(coef_df.round(3))


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

coef_long = coef_df.reset_index().melt(id_vars="index", var_name="Condition", value_name="Weight")
plt.figure(figsize=(10,5))
sns.barplot(data=coef_long, x="Condition", y="Weight", hue="index")
plt.title("Feature Influence per Condition")
plt.ylabel("Coefficient Weight")
plt.xlabel("")
plt.legend(title="Feature", bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()


In [ ]:
# --- Extract Rank (Dominant/Subordinate) ---
df_rank = merged_all[merged_all["Condition"].isin(["Dominant", "Subordinate"])].copy()

# --- Extract Valence (Juvenile/CD1) ---
df_valence = merged_all[merged_all["Condition"].isin(["Juvenile", "CD1"])].copy()


In [ ]:
feature_cols = [
    "allogrooming", "anogenital sniffing", "body sniffing",
    "chasing", "facial sniffing", "fighting", "Resp_MeanRate"
]


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
import pandas as pd

def fit_axis_model(df, label_name):
    X = df[feature_cols].fillna(0)
    y = df["Condition"]
    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    scores = cross_val_score(model, X, y_enc, cv=5, scoring="accuracy")
    model.fit(X, y_enc)

    coefs = model.named_steps["logisticregression"].coef_[0]
    coef_df = pd.DataFrame({
        "Feature": feature_cols,
        "Weight": coefs
    }).set_index("Feature")

    print(f"✅ {label_name} classifier: {le.classes_.tolist()} | Mean acc={scores.mean():.2f} ± {scores.std():.2f}")
    return coef_df


In [ ]:
# Check how many trials per condition
merged_all["Condition"].value_counts()


In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import LeaveOneGroupOut
import warnings

# Choose the features you want the model to consider.
# We'll safely intersect these with what's actually in merged_all.
CANDIDATE_FEATURES = [
    "Resp_MeanRate", "Resp_IBI_std", "Resp_IBI_cv",
    "Resp_Amp_mean", "Resp_Amp_cv", "Resp_Power_total", "Resp_Power_ratioHL",
    "Session_Rate_Hz",
    "allogrooming", "anogenital sniffing", "body sniffing",
    "chasing", "facial sniffing", "fighting", "Posturing"
]


def pick_features(df, candidates):
    """Return (feature_cols, X) after intersecting with available columns and filling NaNs."""
    cols = [c for c in candidates if c in df.columns]
    X = df[cols].astype(float).fillna(0.0)
    return cols, X

def l1_logreg_pipeline():
    """Binary L1 logistic (liblinear supports L1 for binary)."""
    return make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        LogisticRegression(
            penalty="l1", solver="liblinear", max_iter=2000, class_weight=None
        )
    )

def best_n_splits(y_enc, max_cap=5, floor=2):
    """
    Pick a safe #folds for StratifiedKFold given class counts.
    Ensures every fold has at least 1 sample per class.
    """
    counts = np.bincount(y_enc)
    min_per_class = counts.min()
    return int(max(floor, min(min_per_class, max_cap)))

def nonzero_coefs(pipeline, feature_cols):
    """Return a sorted Series of non-zero L1 coefficients (positive → class 1, negative → class 0)."""
    lr = pipeline.named_steps["logisticregression"]
    coefs = pd.Series(lr.coef_.ravel(), index=feature_cols)
    nz = coefs[coefs != 0].sort_values(key=lambda s: s.abs(), ascending=False)
    return nz


In [ ]:
# -------------------------
# RANK DATA (Dom vs Sub)
# -------------------------
df_rank = merged_all[merged_all["Condition"].isin(["Dominant", "Subordinate"])].copy()

# Label encode: Dominant=1, Subordinate=0
df_rank["y"] = (df_rank["Condition"] == "Dominant").astype(int)

# Features
rank_features, X_rank = pick_features(df_rank, CANDIDATE_FEATURES)
y_rank = df_rank["y"].values
groups_rank = df_rank["Subject_ID"].astype(str).values  # for across-subject CV

print(f"Rank: n={len(df_rank)}, subjects={df_rank['Subject_ID'].nunique()}, features={rank_features}")

# ---------- Within-subject CV ----------
cv_ws = StratifiedKFold(
    n_splits=best_n_splits(y_rank, max_cap=5, floor=2),
    shuffle=True, random_state=42
)
rank_model = l1_logreg_pipeline()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ws_scores = cross_val_score(rank_model, X_rank, y_rank, cv=cv_ws, scoring="accuracy")

print(f"[Rank] Within-subject CV accuracy: {ws_scores.mean():.2f} ± {ws_scores.std():.2f} (n_splits={cv_ws.n_splits})")

# ---------- Across-subject CV (leave-one-subject-out) ----------
logo = LeaveOneGroupOut()
accs = []
for train_idx, test_idx in logo.split(X_rank, y_rank, groups=groups_rank):
    if len(np.unique(y_rank[train_idx])) < 2 or len(np.unique(y_rank[test_idx])) < 2:
        # if a fold has only one class, skip (too few samples for that subject)
        continue
    m = l1_logreg_pipeline()
    m.fit(X_rank.iloc[train_idx], y_rank[train_idx])
    yhat = m.predict(X_rank.iloc[test_idx])
    accs.append(accuracy_score(y_rank[test_idx], yhat))

if accs:
    print(f"[Rank] Across-subject (LOSO) accuracy: {np.mean(accs):.2f} ± {np.std(accs):.2f}  (folds={len(accs)})")
else:
    print("[Rank] Across-subject (LOSO): not enough data to compute.")

# ---------- Fit final model on all rank data and show L1-selected features ----------
rank_final = l1_logreg_pipeline().fit(X_rank, y_rank)
nz_rank = nonzero_coefs(rank_final, rank_features)
print("\n[Rank] Non-zero L1 coefficients (→ Dominant if positive):")
display(nz_rank)


In [ ]:
resp_data[trial] = {"signal": sig, "time": t, "sampling_rate": fs}


In [ ]:
import numpy as np
from scipy.signal import find_peaks, welch

def compute_resp_features(signal, time, fs=100):
    """Extract rich respiration features per trial."""
    # Mask any NaNs
    sig = np.nan_to_num(signal)
    
    # --- Basic rate features ---
    dur = time[-1] - time[0]
    peaks, _ = find_peaks(sig, distance=fs * 0.0833)  # ≥12 Hz refractory
    ibi = np.diff(time[peaks]) if len(peaks) > 1 else np.array([np.nan])
    rate = len(peaks) / dur if dur > 0 else np.nan
    
    # --- Variability metrics ---
    ibi_std = np.nanstd(ibi)
    ibi_cv = ibi_std / np.nanmean(ibi) if np.nanmean(ibi) > 0 else np.nan
    
    # --- Amplitude metrics ---
    amp = sig[peaks] if len(peaks) > 0 else np.array([np.nan])
    amp_mean = np.nanmean(amp)
    amp_std = np.nanstd(amp)
    amp_cv = amp_std / amp_mean if amp_mean > 0 else np.nan

    # --- Spectral power metrics (0–10 Hz window) ---
    f, Pxx = welch(sig, fs=fs, nperseg=fs*2)
    band_mask = f < 10
    total_power = np.trapz(Pxx[band_mask], f[band_mask])
    low_band = np.trapz(Pxx[(f>=1) & (f<4)], f[(f>=1) & (f<4)])  # 1–4 Hz
    high_band = np.trapz(Pxx[(f>=4) & (f<8)], f[(f>=4) & (f<8)])  # 4–8 Hz
    ratio_highlow = high_band / low_band if low_band > 0 else np.nan

    return {
        "Resp_MeanRate": rate,
        "Resp_IBI_std": ibi_std,
        "Resp_IBI_cv": ibi_cv,
        "Resp_Amp_mean": amp_mean,
        "Resp_Amp_cv": amp_cv,
        "Resp_Power_total": total_power,
        "Resp_Power_ratioHL": ratio_highlow
    }


| Feature            | Reflects                       | Relevance to Rank/Valence                                   |
| ------------------ | ------------------------------ | ----------------------------------------------------------- |
| Resp_MeanRate      | Overall arousal/sniffing tempo | Higher in positive (appetitive) conditions → active investigation and approach; lower in negative (aversive) conditions → freezing, vigilance, or inhibition.    |
| Resp_IBI_std / CV  | Temporal irregularity          | Instability during stress or approach/avoidance transitions |
| Resp_Amp_mean / CV | Breath depth and variability   | Deep steady breaths in calm/social dominance                |
| Resp_Power_total   | Global rhythmicity strength    | Stronger oscillatory drive during engagement                |
| Resp_Power_ratioHL | Relative sniffing mode         | Fast (4–8 Hz) = exploration/stress; slow (1–4 Hz) = calm    |


In [ ]:
resp_features_list = []

for trial, rec in resp_data.items():
    feats = compute_resp_features(rec["signal"], rec["time"], rec["sampling_rate"])
    feats["Trial"] = trial
    resp_features_list.append(feats)

resp_feat_df = pd.DataFrame(resp_features_list)
print(resp_feat_df.head())


In [ ]:
merged_all = merged_all.merge(resp_feat_df, on="Trial", how="left")
